In [1]:
"""
COVID Analysis Notebook
-----------------------
This notebook reproduces the experiments used in the related article.
All random seeds and logic are preserved for reproducibility.
"""

'\nCOVID Analysis Notebook\n-----------------------\nThis notebook reproduces the experiments used in the related article.\nAll random seeds and logic are preserved for reproducibility.\n'

# importation

In [2]:
# Core
import os
import re
import warnings
warnings.filterwarnings("ignore")

# Data & numerics
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics & signal processing
from scipy.stats import (
    ttest_ind, mannwhitneyu, zscore, pearsonr
)
from scipy.signal import find_peaks
from scipy.spatial.distance import euclidean

# Machine learning
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score,
    confusion_matrix, mean_squared_error
)
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import DBSCAN
from sklearn.pipeline import Pipeline 

# Utilities
from tqdm import tqdm
import joblib


# fonctions

In [3]:
class TopFeatureSelectorByMannWhitney(BaseEstimator, TransformerMixin):
    def __init__(self, n_features=100, top_gini=1000):
        self.n_features = n_features
        self.top_gini = top_gini

    def fit(self, X, y):
        rf = RandomForestClassifier(n_estimators=1000, random_state=0)
        rf.fit(X, y)
        importances = rf.feature_importances_
        top_indices = np.argsort(importances)[::-1][:self.top_gini]

        mw_scores = self._mann_whitney_score(X[:, top_indices], y)
        mw_norm = MinMaxScaler().fit_transform(mw_scores.reshape(-1, 1)).ravel()
        lds_scores = self._linear_discriminative_score(X[:, top_indices], y)
        lds_norm = MinMaxScaler().fit_transform(lds_scores.reshape(-1, 1)).ravel()

        combined_scores = [(top_indices[i], (lds_norm[i]+ mw_norm[i]) / 2)  for i in range(len(top_indices))]#
        combined_sorted = sorted(combined_scores, key=lambda x: x[1], reverse=True)

        self.indices_ = [idx for idx, score in combined_sorted[:self.n_features]]
        return self

    def transform(self, X):
        return X[:, self.indices_]

    def _linear_discriminative_score(self, X, y):
        class_labels = np.unique(y)
        idx0 = y == class_labels[0]
        idx1 = y == class_labels[1]
        mean0 = X[idx0].mean(axis=0)
        mean1 = X[idx1].mean(axis=0)
        var0 = X[idx0].var(axis=0)
        var1 = X[idx1].var(axis=0)
        return (mean1 - mean0) ** 2 / (var1 + var0 + 1e-8)

    def _mann_whitney_score(self, X, y):
        scores = []
        for i in range(X.shape[1]):
            try:
                u_stat, _ = mannwhitneyu(X[y == 0, i], X[y == 1, i], alternative='two-sided')
            except ValueError:
                u_stat = 0
            scores.append(u_stat)
        return np.array(scores)

# step1 : find the bin size

In [4]:
nbVariable=100
nEstimatorRF=100

results_list = []

for file in ['./covid_10MZ.csv','./covid_5MZ.csv','./covid_0.5MZ.csv']:
    print(f"File : {file}")
    df = pd.read_pickle(file)

    df['echantillon']=df['echantillon'].apply(lambda x : x.split('/')[3].split('_')[2])
    df['categ']=df['categ'].apply(lambda x : x.split('/')[2])
    df=df[df['categ']!='Mild']
    df['categ']=df['categ'].apply(lambda x : 0 if x=='Negative' else 1)
    df=df.drop_duplicates(subset=['echantillon'])

    a = int(len(df['output_vector'].iloc[0]) / 3)

    merged_df=df.copy()
    X = np.array([np.array(vec) for vec in merged_df['output_vector']])
    Y=merged_df['categ'].values

    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=2024)

    pipeline = Pipeline([
    ('feature_selection', TopFeatureSelectorByMannWhitney(n_features=nbVariable, top_gini=1000)),
    ('classification', RandomForestClassifier(n_estimators=nEstimatorRF, random_state=0))
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=10)
    mean_cv_score = scores.mean()
    std_cv_score = scores.std()

    results_list.append({
        'Fichier': file,
        'Accuracy': mean_cv_score,
        'std': std_cv_score
    })

results_df = pd.DataFrame(results_list)

File : ./covid_10MZ.csv


NameError: name 'Pipeline' is not defined

In [ ]:
results_df['mz_size']=results_df['Fichier'].apply(lambda x : 10 if '_10' in x else 5 if '_5' in x else 0.5)

grouped = results_df.groupby('mz_size').agg({
    'Accuracy': 'mean',
    'std': 'mean'
}).reset_index().sort_values('mz_size')

plt.figure(figsize=(12, 8))

plt.errorbar(grouped['mz_size'], grouped['Accuracy'],
             yerr=grouped['std'],
             fmt='-o', color='tab:blue', linewidth=2, markersize=8,
             capsize=5, elinewidth=1.5, ecolor='gray',
             label='Mean CV accuracy ± 1 STD')

plt.xlabel('M/z bin size', fontsize=10, labelpad=10)
plt.ylabel('Mean cross-validation accuracy on training set', fontsize=10, labelpad=10)
plt.legend(loc='lower right', fontsize=10, frameon=True)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# step2 : training model

In [ ]:
df=pd.read_pickle('./covid_0.5MZ.csv')
df['echantillon']=df['echantillon'].apply(lambda x : x.split('/')[3].split('_')[2])
df['categ']=df['categ'].apply(lambda x : x.split('/')[2])
df=df[df['categ']!='Mild']
df['categ']=df['categ'].apply(lambda x : 0 if x=='Negative' else 1)
df=df.drop_duplicates(subset=['echantillon'])
a=int(len(df['output_vector'].iloc[0])/3)
merged_df=df.copy()
X = np.array([np.array(vec) for vec in merged_df['output_vector']])
Y=merged_df['categ'].values
X_train=X
y_train=Y
selector = TopFeatureSelectorByMannWhitney(n_features=1000, top_gini=1000)
selector.fit(X_train, y_train)
indices_discriminants = selector.indices_

In [ ]:
indices_discriminants

In [ ]:
a

# step3 : analyse

In [ ]:
# For example, we analyze the bin number 16044, which corresponds to C19-sphingosine.
# You can change this value according to the discriminative bin numbers mentioned in the paper.
#
# Note that the vector is composed as follows:
# - first: the intensity values up to a given index a,
# - then: the m/z (mass-to-charge) values,
# - and finally (from -a to the end): the RT (retention time) values.
#
# So, for example, if a < indices_discriminants[1] < a + a,
# it means that indices_discriminants[1] corresponds to an m/z value.
# Therefore, to plot intensity versus m/z, the user must select:
#   index_2 = indices_discriminants[1] - a

index_1 = 16044
index_2 = 16044 + a

#valine : 
#index_1 = indices_discriminants[3]
#index_2 = indices_discriminants[3] + a



In [ ]:
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 10,
    'axes.labelsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'legend.title_fontsize': 10
})
feature_values_1 = [x[index_1] for x in X_train]
feature_values_2 = [x[index_2] for x in X_train]

start_time = 2
end_time = 20
mz_min = 100
mz_max = 700

feature_values_2 = [(val * (mz_max - mz_min)) + mz_min for val in feature_values_2]


index_1 = 'm/z'
index_2 = 'Normalized intensity'

# Catégories
y_train2 = ['Controle' if x == 0 else 'Severe Cases' for x in y_train]



# Création du DataFrame
df_plot = pd.DataFrame({
    f'{index_1}': feature_values_2,
    f'{index_2}': feature_values_1,
    'Category': y_train2
})

# Figure
plt.figure(figsize=(14, 10))
scatter = sns.scatterplot(
    data=df_plot,
    x=index_1,
    y=index_2,
    hue='Category',
    palette='tab10',
    s=120,
    edgecolor='k',
    alpha=0.8
)

plt.xlabel(index_1, labelpad=14)
plt.ylabel(index_2, labelpad=14)
plt.legend(title='Category', loc='upper left', frameon=True)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:

data=np.array(feature_values_2)
X = np.array(feature_values_2).reshape(-1, 1)

db = DBSCAN(eps=0.01, min_samples=3).fit(X)  # eps est la tolérance (à ajuster)
labels = db.labels_

clusters = {}
for label in np.unique(labels):
    if label == -1:
        continue  # -1 = bruit, on ignore
    cluster_data = data[labels == label]
    mean = np.mean(cluster_data)
    std = np.std(cluster_data)
    std_ppm = (std / mean) * 1e6
    clusters[label] = {
        "count": len(cluster_data),
        "mean": mean,
        "std": std,
        "std_ppm": std_ppm
    }

df2 = pd.DataFrame.from_dict(clusters, orient='index')
df2.index.name = "Cluster"
print(df2)

In [ ]:
'''
boxPlot for C19-sphingosine
borne_min = 322.26
borne_max = 333
feature_values_2=np.array(feature_values_2)
y_train2 = np.array(y_train2)

valeurs_filtrees = feature_values_2[(feature_values_2 >= borne_min) & (feature_values_2 <= borne_max)]
indices_dans_borne = np.where((feature_values_2 >= borne_min) & (feature_values_2 <= borne_max))[0]


print(len(valeurs_filtrees))
moyenne = np.mean(valeurs_filtrees)
ecart_type = np.std(valeurs_filtrees)

print("Moyenne :", moyenne)
print("Écart-type :", ecart_type)
categories_dans_borne = y_train2[indices_dans_borne]

counts = pd.Series(categories_dans_borne).value_counts()

print("Comptage des catégories dans la plage :")
print(counts)

gg=np.array(feature_values_1)
gg=gg[indices_dans_borne]

sns.boxplot(x=categories_dans_borne, y=gg, showfliers=False)
'''